In [1]:
!pip -q install diffusers transformers accelerate gradio google-genai pillow
from google import genai
client = genai.Client(api_key="AQ.Ab8RN6LwLFISQUeyEgXMF7ga5SuBjxa8bI-89qKRpqtG00wX7A")
for m in client.models.list():
    if "gemma" in m.name.lower(): print(m.name)

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\Muskan\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\Muskan\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
  File "C:\Users\Muskan\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
  File "C:\Users\Muskan\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 90, in read
    data = self.__fp.read(amt)
  File "C:\Users\Muskan\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 466, in read
    s = self.fp.read(amt)
  File "C:\Users\Muskan\AppData\Local\Programs\Python\Python310\lib\socket.py", line 705, in r

ImportError: cannot import name 'genai' from 'google' (unknown location)

In [ ]:
!pip uninstall -y diffusers
!rm -rf /usr/local/lib/python3.13/dist-packages/diffusers /usr/local/lib/python3.13/dist-packages/diffusers-*
!pip install --no-cache-dir diffusers transformers accelerate gradio google-genai pillow

Found existing installation: diffusers 0.42.0
Uninstalling diffusers-0.42.0:
  Successfully uninstalled diffusers-0.42.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 35.3 MB/s  0:00:00


In [ ]:
from diffusers import AutoPipelineForImage2Image
print("ok")

ok


In [ ]:
from getpass import getpass
from google import genai
client = genai.Client(api_key=getpass("API key: AQ.Ab8RN6LwLFISQUeyEgXMF7ga5SuBjxa8bI-89qKRpqtG00wX7A ").strip())
MODEL = "gemma-4-26b-a4b-it"
print("client ready")

API key: AQ.Ab8RN6LwLFISQUeyEgXMF7ga5SuBjxa8bI-89qKRpqtG00wX7A ··········
client ready


In [ ]:
import json, re, torch, gradio as gr
from PIL import Image

pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")

RULES = """You help young kids turn scribbles into illustrations.
Look at the drawing. Reply with ONLY JSON, no markdown:
{"refuse": false, "caption": "what the child probably drew, one short friendly sentence", "prompt": "a child-safe, cheerful description for an image generator"}
If the drawing or any text in it is violent, scary, sexual, hateful or inappropriate, reply:
{"refuse": true, "caption": "", "prompt": ""}"""

STYLES = {"Storybook": "children's storybook illustration, soft colors",
          "Crayon": "bright crayon drawing on paper",
          "Watercolor": "gentle watercolor painting"}

def to_img(sk):
    im = sk["composite"] if isinstance(sk, dict) else sk
    im = im.convert("RGBA")
    bg = Image.new("RGBA", im.size, "white"); bg.alpha_composite(im)
    return bg.convert("RGB").resize((512, 512))

def interpret(sk):
    if sk is None: return "Draw something first.", ""
    r = client.models.generate_content(model=MODEL, contents=[to_img(sk), RULES])
    t = re.sub(r"```(json)?", "", r.text).strip()
    try: j = json.loads(t)
    except Exception: return "Gemma gave bad JSON: " + t[:200], ""
    if j.get("refuse"): return "Let's draw something nice instead!", ""
    return j["caption"], j["prompt"]

def render(sk, prompt, style, strength):
    if sk is None or not prompt: return None
    return pipe(prompt=f"{prompt}, {STYLES[style]}", image=to_img(sk),
                strength=strength, num_inference_steps=4, guidance_scale=0.0).images[0]

with gr.Blocks() as demo:
    gr.Markdown("## Scribble to Story, powered by Gemma 4")
    sk = gr.Sketchpad(type="pil", label="Draw here")
    seen = gr.Textbox(label="I think you drew...")
    prompt = gr.Textbox(label="Image prompt (editable)")
    style = gr.Dropdown(list(STYLES), value="Storybook", label="Style")
    strength = gr.Slider(0.4, 0.8, 0.6, label="How much to keep your drawing")
    gr.Button("1. What did I draw?").click(interpret, sk, [seen, prompt])
    out = gr.Image(label="Result")
    gr.Button("2. Make it real").click(render, [sk, prompt, style, strength], out)
demo.launch(share=True, debug=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl_img2img.py:898: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_proc

  0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl_img2img.py:898: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://03d6570262fccededc.gradio.live
